# 🛡️ Misinformation Detection with NLP
### Exploring AI-based approaches for civic integrity in African digital spaces

**Author:** Pantouin Adjinsala  
**Affiliation:** University Lecturer & Civic Tech Contributor, AfricTivistes CitizenLab Cameroon  
**Context:** This notebook is part of ongoing exploration into how AI and NLP can be applied to detect misinformation — a growing challenge in African digital and civic spaces, particularly around elections, health, and governance.

---

## 🎯 Objective
Build a baseline text classification model capable of distinguishing **reliable** from **unreliable** statements using the LIAR dataset — a benchmark dataset in misinformation research. The goal is not just technical accuracy, but to demonstrate how such a pipeline could be adapted to African language contexts and civic monitoring use cases.

---

## 📌 Contents
1. Setup & Imports
2. Dataset Loading & Overview
3. Exploratory Data Analysis
4. Text Preprocessing
5. Feature Extraction (TF-IDF)
6. Model Training & Evaluation
7. Visualisations
8. Discussion & Next Steps

## 1. Setup & Imports

In [ ]:
# Install datasets library for loading from HuggingFace
!pip install datasets -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('✅ All libraries loaded successfully.')

## 2. Dataset Loading & Overview

The **LIAR dataset** (Wang, 2017) contains ~12,800 short political statements labelled across 6 veracity levels: *pants-fire, false, barely-true, half-true, mostly-true, true*.

For this binary classification task, we simplify to:
- **0 — Fake:** `pants-fire`, `false`, `barely-true`
- **1 — Real:** `half-true`, `mostly-true`, `true`

In [ ]:
# Load dataset from HuggingFace
raw = load_dataset('liar')

def to_df(split):
    df = pd.DataFrame(raw[split])
    # Binary label mapping
    fake_labels = {'pants-fire', 'false', 'barely-true'}
    df['binary_label'] = df['label'].apply(
        lambda x: 0 if raw[split].features['label'].int2str(x) in fake_labels else 1
    )
    df['label_name'] = df['label'].apply(
        lambda x: raw[split].features['label'].int2str(x)
    )
    df['binary_label_name'] = df['binary_label'].map({0: 'Fake', 1: 'Real'})
    return df

train_df = to_df('train')
test_df  = to_df('test')
val_df   = to_df('validation')

print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows | Val: {len(val_df):,} rows')
train_df[['statement', 'label_name', 'binary_label_name']].head(8)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Original 6-class distribution ---
label_order = ['pants-fire','false','barely-true','half-true','mostly-true','true']
counts_6 = train_df['label_name'].value_counts().reindex(label_order)
colors_6 = ['#d62728','#ff7f0e','#ffbb78','#98df8a','#2ca02c','#1f77b4']
axes[0].bar(counts_6.index, counts_6.values, color=colors_6, edgecolor='white')
axes[0].set_title('Original Label Distribution (Train)', fontweight='bold')
axes[0].set_xlabel('Veracity Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# --- Binary distribution ---
counts_2 = train_df['binary_label_name'].value_counts()
axes[1].pie(
    counts_2.values, labels=counts_2.index,
    autopct='%1.1f%%', startangle=90,
    colors=['#ff7f0e','#1f77b4'], wedgeprops={'edgecolor':'white','linewidth':1.5}
)
axes[1].set_title('Binary Label Distribution (Train)', fontweight='bold')

plt.tight_layout()
plt.savefig('label_distribution.png', bbox_inches='tight')
plt.show()
print(counts_2.to_string())

In [ ]:
# Statement length analysis
train_df['statement_length'] = train_df['statement'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [('Fake','#ff7f0e'), ('Real','#1f77b4')]:
    subset = train_df[train_df['binary_label_name'] == label]['statement_length']
    ax.hist(subset, bins=40, alpha=0.6, label=label, color=color, edgecolor='white')
ax.set_title('Statement Length Distribution by Label', fontweight='bold')
ax.set_xlabel('Word Count')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('length_distribution.png', bbox_inches='tight')
plt.show()

print(train_df.groupby('binary_label_name')['statement_length'].describe().round(1))

## 4. Text Preprocessing

We apply lightweight preprocessing — lowercasing and punctuation removal. We intentionally avoid aggressive stemming at this stage to preserve interpretability of feature weights later.

In [ ]:
import re

def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # remove punctuation & numbers
    text = re.sub(r'\s+', ' ', text).strip() # collapse whitespace
    return text

for df in [train_df, test_df, val_df]:
    df['clean_statement'] = df['statement'].apply(preprocess)

# Sanity check
train_df[['statement','clean_statement']].head(3)

## 5. Feature Extraction — TF-IDF

**TF-IDF** (Term Frequency–Inverse Document Frequency) converts text into numerical vectors by weighting words that are frequent in a document but rare across the corpus — making it well-suited to detecting language patterns associated with unreliable claims.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),      # unigrams + bigrams
    stop_words='english',
    min_df=3                 # ignore very rare terms
)

X_train = vectorizer.fit_transform(train_df['clean_statement'])
X_test  = vectorizer.transform(test_df['clean_statement'])
X_val   = vectorizer.transform(val_df['clean_statement'])

y_train = train_df['binary_label'].values
y_test  = test_df['binary_label'].values
y_val   = val_df['binary_label'].values

print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')
print(f'Train matrix shape: {X_train.shape}')

## 6. Model Training & Evaluation

We use **Logistic Regression** — a strong, interpretable baseline for text classification. Interpretability matters here: in a civic/governance context, understanding *why* a model flags a statement is as important as the prediction itself.

In [ ]:
model = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42)
model.fit(X_train, y_train)

# Cross-validation on training set
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
print(f'5-Fold CV F1 (train): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

# Test set evaluation
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print('\n--- Test Set Classification Report ---')
print(classification_report(y_test, y_pred, target_names=['Fake','Real']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_prob):.3f}')

## 7. Visualisations

In [ ]:
# --- Confusion Matrix ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Fake','Real'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Test Set', fontweight='bold')

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='#1f77b4', lw=2, label=f'ROC curve (AUC = {auc:.2f})')
axes[1].plot([0,1],[0,1],'--', color='grey', lw=1, label='Random baseline')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('model_evaluation.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Top predictive features ---
feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_[0]
top_n = 15

top_fake_idx = np.argsort(coefs)[:top_n]          # most negative = Fake
top_real_idx = np.argsort(coefs)[-top_n:][::-1]   # most positive = Real

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, idx, title, color in [
    (axes[0], top_fake_idx, 'Top 15 — Fake Indicators', '#d62728'),
    (axes[1], top_real_idx, 'Top 15 — Real Indicators', '#2ca02c')
]:
    words  = [feature_names[i] for i in idx]
    values = [abs(coefs[i]) for i in idx]
    ax.barh(words[::-1], values[::-1], color=color, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Coefficient Magnitude')

plt.suptitle('Most Predictive Words & Bigrams by Class', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('top_features.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- Live prediction demo ---
def predict_statement(text):
    clean = preprocess(text)
    vec   = vectorizer.transform([clean])
    pred  = model.predict(vec)[0]
    prob  = model.predict_proba(vec)[0]
    label = 'REAL ✅' if pred == 1 else 'FAKE ⚠️'
    print(f'Statement : "{text}"')
    print(f'Prediction: {label}')
    print(f'Confidence: Fake={prob[0]:.2%} | Real={prob[1]:.2%}\n')

# Test with sample statements
predict_statement("The government has increased education funding by 20% this year.")
predict_statement("Vaccines contain microchips designed to track citizens.")
predict_statement("Unemployment has fallen to its lowest level in a decade.")

## 8. Discussion & Next Steps

### What we built
A fully functional NLP pipeline for binary misinformation detection using TF-IDF features and Logistic Regression, achieving competitive baseline performance on the LIAR benchmark.

### Limitations
- **Dataset bias:** LIAR is drawn from US political fact-checking. Performance on African political or health misinformation may differ significantly.
- **Language coverage:** The current model handles English only. Francophone African contexts (French, Camfranglais, pidgin) are not yet covered.
- **Context-blind:** Short statements lose important context (speaker identity, platform, date). Speaker metadata in LIAR is not yet exploited.

### Planned Extensions
1. **French-language adaptation** — retrain on French-language fact-checked claims from sources like AfricaCheck and Désinfox Afrique
2. **Feature enrichment** — incorporate speaker credibility scores and subject metadata from the LIAR dataset
3. **Transformer upgrade** — fine-tune a multilingual model (e.g. `xlm-roberta-base`) for cross-lingual transfer to African language contexts
4. **Deployment** — wrap the pipeline in a lightweight API for integration into civic monitoring dashboards

---

> *This work is part of broader research into AI applications for civic integrity and digital governance in Africa.*  
> *Contributions and collaborations welcome.*

**References**
- Wang, W. Y. (2017). Liar, Liar Pants on Fire: A New Benchmark Dataset for Fake News Detection. *ACL 2017*.
- AfricaCheck — https://africacheck.org
- AfricTivistes — https://africTivistes.org